<a href="https://colab.research.google.com/github/ShiX27/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last = None
    for attempt in range(tries):      # TODO: loop up to `tries` times
        try:      # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
          r = requests.get(BASE, params=params, timeout=15)
          if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])
          if r.status_code not in RETRY_STATUSES:   # TODO:   raise on a status that is not in RETRY_STATUSES
            raise RuntimeError(f'not retryable: {r.status_code} {r.text[:200]}')
          last = f'status{r.status_code}'
        except requests.RequestException as e:
            last = repr(e)
        if attempt == tries - 1:
            break
        delay = 2 ** attempt
        print(f'Attempt {attempt + 1} failed ({last}); retrying in {delay}s')
        time.sleep(delay)       # TODO:   otherwise sleep (increasing each time) and try again
    raise RuntimeError(f'Failed after {tries} attempts: {last}')    # TODO: raise if every attempt failed

**Code Explanation:** First, import `requests`, `pandas` as `pd`, and `time`. Define `BASE` as the Open-Meteo API endpoint and `RETRY_STATUSES` as the set of HTTP status codes (`429, 500, 502, 503, 504`) that allow retries. Then, define the `get_weather()` function with the start and end dates, lattitude and longititude, and a maxmimum of 3 attempts. Create a `params` dictionary containing the coordinates, date range, daily maximum temperature, total precipitation, and time zone.

Next, initialize `last = None` to store information about the most recent failure. Use a `for` loop that repeats up to `tries` times to attempt the API request. Start the `try` block, send a `GET` request to the API endpoint stored in `BASE`, pass the `params` dictionary to the API, and set a 15-second timeout. Then, check whether the API request succeeded. If the status code is 200, convert the `daily` block of the JSON response insto a Pandas DataFrame and return it, which immediately ends the function. If the status code is not in `RETRY_STATUSES`, raise a `RuntimeError` immediately without retrying. Otherwise, sotre the retryable status code in `last` so the program can retry if attempts remain.

If a request exception occurs, use the `except` block to catch the exception and store the error information in `last` using `repr(e)`. Next, check whether the current attempt is the final attempt using `if attempt == tries - 1`, and use `break` to exit the loop if no attempts remain. Otherwise, calculate an increasing delay using `2 ** attempt`, print a message showing the failed attempt, the last recorded failure, and the waiting time. Use `time.sleep(delay)` to pause before trying again. Finally, if all attempts fail, raise a `RuntimeError` showing the total number of attempts and the last recorded failure.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates, then keep just the two dates you care about
try:
    wx = get_weather('2026-09-05', '2026-10-03')
    retrieved_on = pd.Timestamp.now(tz='America/New_York')
    wx = wx.loc[wx['time'].isin(['2026-09-05', '2026-10-03'])].copy()
    used_fallback = False
    display(wx)

except RuntimeError as e:
    if 'not retryable' in str(e).lower():
        raise
    print('API request failed:', e)
    used_fallback = True

,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
28,2026-10-03,22.0,14.7


**Code Explanation:** First, start the `try` block, retrieve the weather data for the date range from 2026-09-05 to 2026-10-03 using the `get_weather()` function defined in Part 1, and store the data in `wx`. Second, record the current date and time immediately after the successful API request using the `America/New_York` timezone, and store it in retrieved_on. Third, filter the data to keep only the two requested dates (2026-09-05 and 2026-10-03) from the time column, and store a copy of the filtered data in `wx`. If the API request succeeds, set `used_fallback = False` and display the `wx` DataFrame. If a `RuntimeError `occurs, use the `except` block to catch the error. If the error message contains `'not retryable'`, raise the error again without using fallback data. Otherwise, print the API failure message and set `used_fallback = True` to indicate that fallback data should be used in Part 3.

In this case, API request succeeded, so the output displays the `wx` DateFrame containing real API weather data for the two requested dates.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
if used_fallback == True:
  wx = FALLBACK.copy()
  wx['source'] = 'fallback sample'
else:
  wx['source'] = 'open-meteo'

wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


**Code Explanation:** First create the `FALLBACK` DataFrame with sample weather data for the two requested dates. Second, if `used_fallback` from Part 2 is `True`, store a copy of the `FALLBACK` DataFrame in `wx` and add a `source` column with the value `'fallback sample'`. Otherwise, if `used_fallback` is `False`, add a `source` column with the value `'open-meteo'` to the existing `wx` DataFrame. Finally, display the `wx` DataFrame. Since the API request succeeded in Part 2, the output contains real API weather data for the two requested dates, with the `source` column labeled `'open-meteo'`.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
print(wx.dtypes)
wx

date         datetime64[ns]
temp_max            float64
precip_mm           float64
source               object
dtype: object


,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


**Code Explanation:** First, rename the columns `time` to `date`, `temperature_2m_max` to `temp_max`, and `precipitation_sum` to `precip_mm`, and store the result in the `wx` DataFrame. Second, convert the `date` column from strings to the `datetime` data type. Finally, print the data type of each column in `wx` and display the `wx` DataFrame.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


**Code Explanation:** Uncomment the `assert` statements to check that the `wx` DataFrame has the required column names (`date`, `temp_max`, `precip_mm`, and `source`), contains exactly 2 rows, and has a `date` column with a `datatime` data type. Also, check that the `precip_mm` column contains no missing values (NaN) and no negative precipitation values. If all assertions pass, print `'weather table is safe to join.'`

### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
wetter_date = wx.loc[wx['precip_mm'].idxmax(), 'date'].date()
difference = wx['precip_mm'].max() - wx['precip_mm'].min()
print(f'{wetter_date} was wetter by {difference:.1f} mm of precipitation.')

2026-10-03 was wetter by 12.2 mm of precipitation.


2026-10-03 was wetter by 12.2 mm of precipitation.

**Code Explanation:** First, use `.idxmax()` to find the index of the row with the highest `precip_mm` value in the `wx` DataFrame, and use `.loc[]` to retrieve the corresponding date. Then, use `.date()` to  remove the time poriton (00:00:00) and store the result in `wetter_date`. Second, calculate the difference beteween the maxmimum and minimum precipitation vlaues in `wx` using `.max()` and `.min()`, and store the result in `difference`. Finally, print a sentence stating which date was wetter and by how many millimeters of precipitation, formatting the difference to one decimal place.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
print('Endpoint:', BASE)
print('Coordinates: (38.03, -78.51)')
print('Date range: 2026-09-05 to 2026-10-03')
print('Retrieved on:', retrieved_on.strftime('%Y-%m-%d %H:%M:%S %Z'))
print('Data source:', 'fallback sample' if used_fallback else 'open-meteo')

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: (38.03, -78.51)
Date range: 2026-09-05 to 2026-10-03
Retrieved on: 2026-10-11 15:03:05 EDT
Data source: open-meteo


**Code Explanation:** Print the API endpoint using the URL stored in `BASE`. Print the coordinates as (latitude, longitude), which are `(38.03, -78.51)`. Print the date range from `2026-09-05 to 2026-10-03`. Print the date and time when the API data was retrieved, using the `retrieved_on` variable from Part 2 and including the Eastern Time zone abbreviation (EDT). Print the data source as 'fallback sample' if `used_fallback` is `True`; otherwise, print `'open-meteo'`.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If `get_weather()` returned an empty DataFrame on failure instead of raising an error, the program would continue running without indicating that the weather data is missing or the API request failed. If we use an inner join to merge it with 200 days of sales data, none of the sales records would match the empty weather DataFrame, so the result would be an empty DataFrame with zero rows. If we use a left join, all 200 sales days would remain, but the weather values would be missing (NaN). After merging the datasets, we would chart precipitation against revenue. However, because the weather data is missing, the chart might appear empty or omit observations with missing precipitation values. Nobody would catch the problem because no error was raised, the code would continue running, and someone might assume the chart reflects valid data without checking for missing values or the number of rows in the merged dataset.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch
orlando = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
orlando = orlando.loc[orlando['time'].isin(['2026-09-05', '2026-10-03'])].copy()
orlando = orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
orlando['date'] = pd.to_datetime(orlando['date'])
orlando['source'] = 'open-meteo'

# Add a city to both DataFrames
wx['city'] = 'Charlotteville'
orlando['city'] = 'Orlando'

# Combine the two cities
combined = pd.concat([wx, orlando], ignore_index=True)
combined

,date,temp_max,precip_mm,source,city
0,2026-09-05,29.4,2.5,open-meteo,Charlotteville
1,2026-10-03,22.0,14.7,open-meteo,Charlotteville
2,2026-09-05,32.1,3.9,open-meteo,Orlando
3,2026-10-03,30.4,14.5,open-meteo,Orlando


**Code Explanation:** First, retrieve Orlando's weather data for the date range from 2026-09-05 to 2026-10-03 using the coordinates (28.54, -81.38),and store the data in `orlando`. Second, filter the data to keep only the two requested dates (2026-09-05 and 2026-10-03) from the `time` column, and store a copy of the filtered data in `orlando`. Third, rename the columns `time` to `date`, `temperature_2m_max` to `temp_max`, and `precipitation_sum` to `precip_mm`. Fourth, convert the `date` column from strings to the `datetime` data type. Fifth, add a source column with the value 'open-meteo' to the `orlando` DataFrame because the Orlando API request succeeded. Sixth, add a `city` column to both DataFrames to identify Charlottesville and Orlando. Then, use `pd.concat()` to combine the two DataFrames into one long-format DataFrame, with `ignore_index=True` to create a new sequential index. Finally, display the `combined` DataFrame.